In [9]:
import os
import torch
import numpy as np
import pandas as pd
from typing import Dict, List
import time
from dotenv import load_dotenv

from langchain_core.messages import ChatMessage
from langchain_core.prompts import ChatMessagePromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_text_splitters.base import TextSplitter

from financerag.common import get_query_and_retrieved_corpus_text, process_retrieval_df, get_final_result 
from financerag.task import *
from financerag.retrieval import DenseRetrieval
from financerag.hipporag import HippoRAG
from financerag.retrieval import BM25, BM25_Retriever
from financerag.rerank import CrossEncoderReranker

from sentence_transformers import CrossEncoder
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')


In [10]:
load_dotenv(".env")
GG_API_KEY = os.environ.get('GOOGLE_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')

In [ ]:
4

In [11]:
def load_query(dataset_name : str, new_path_to_query = None) -> Dict[str, str]:
    query_df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_queries.jsonl/queries.jsonl", lines = True)
    # Convert into Dict[str, str]
    query_dict = {row['_id'] : row['text'] for i, row in query_df.iterrows()}
    return query_dict

    
def load_corpus(dataset_name : str, new_path_to_corpus = None) -> Dict[str, str]:
    corpus_df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_corpus.jsonl/corpus.jsonl", lines = True)
    # Convert into Dict[str, str]
    corpus_dict = {row['_id'] : row['text'] for i, row in corpus_df.iterrows()}
    return corpus_dict

In [1]:
print('hello world')

hello world


In [2]:
print('hello world')

hello world


In [12]:
# Set up vector database
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004", request_options = {'timeout' : 100000})
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [13]:
def get_vector_store():
    embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004",  request_options = {'timeout' : 100000})
    index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))
    
    vector_store = FAISS(
        embedding_function=embeddings,
        index=index,
        docstore=InMemoryDocstore(),
        index_to_docstore_id={},
    )
    return vector_store

In [14]:
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
device

'mps'

In [15]:
# Get dataset name first
import os
dataset_names = []
for f in os.listdir('finance_dataset'):
    if f.endswith("tsv"):
       dataset_names.append(f.split('_')[0])
dataset_names  

['MultiHeirtt',
 'FinQA',
 'FinanceBench',
 'ConvFinQA',
 'FinQABench',
 'TATQA',
 'FinDER']

In [16]:
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 300
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

In [17]:
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

In [5]:
print('hell world how are you ")

SyntaxError: unterminated string literal (detected at line 1) (4135350098.py, line 1)

In [2]:
4

4

In [19]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    print(f"{dataset_name} task")
    {task_variable} = {dataset_name}Task()
    {task_variable}.load()
    vector_store = get_vector_store()
    {dataset_name}_hippo_rag = HippoRAG(retriever = vector_store)
    {dataset_name}_hippo_rag.offline_indexing(corpus = {task_variable}.corpus, batch_size = 8)
    {task_variable}_result = {dataset_name}_hippo_rag.retrieve(queries = {task_variable}.queries, corpus = {task_variable}.corpus, top_k = 10)
    # {task_variable}_reranker = CrossEncoderReranker(queries = {task_variable}.queries, corpus = {task_variable}.corpus, reranker = model)
    # {task_variable}_final_result = {task_variable}_reranker.rerank(retrieved_result = {task_variable}_result, top_k = 10)
    {task_variable}.save_retrieved_results({task_variable}_result, method_name = 'hippo_rag_baseline')
    """
    print(script_string)


    # MultiHeirtt Task
    print(f"MultiHeirtt task")
    multiheirtt_task = MultiHeirttTask()
    multiheirtt_task.load()
    vector_store = get_vector_store()
    MultiHeirtt_hippo_rag = HippoRAG(retriever = vector_store)
    MultiHeirtt_hippo_rag.offline_indexing(corpus = multiheirtt_task.corpus, batch_size = 8)
    multiheirtt_task_result = MultiHeirtt_hippo_rag.retrieve(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, top_k = 10)
    # multiheirtt_task_reranker = CrossEncoderReranker(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, reranker = model)
    # multiheirtt_task_final_result = multiheirtt_task_reranker.rerank(retrieved_result = multiheirtt_task_result, top_k = 10)
    multiheirtt_task.save_retrieved_results(multiheirtt_task_result, method_name = 'hippo_rag_baseline')
    

    # FinQA Task
    print(f"FinQA task")
    finqa_task = FinQATask()
    finqa_task.load()
    vector_store = get_vector_store()
    FinQA_hippo_rag = Hippo

In [1]:
6

6

In [1]:
print('hello worl')
helo worl dmy name is khair i'm curenlt living in viet nam and what I love doing is leawrning new things everyday to keep my mind sharp

SyntaxError: unterminated string literal (detected at line 2) (3820080120.py, line 2)

In [20]:
# MultiHeirtt Task
print(f"MultiHeirtt task")
multiheirtt_task = MultiHeirttTask()
multiheirtt_task.load()
vector_store = get_vector_store()
MultiHeirtt_hippo_rag = HippoRAG(retriever = vector_store)
MultiHeirtt_hippo_rag.offline_indexing(corpus = multiheirtt_task.corpus, batch_size = 8)
multiheirtt_task_result = MultiHeirtt_hippo_rag.retrieve(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, top_k = 10)
# multiheirtt_task_reranker = CrossEncoderReranker(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, reranker = model)
# multiheirtt_task_final_result = multiheirtt_task_reranker.rerank(retrieved_result = multiheirtt_task_result, top_k = 10)
multiheirtt_task.save_retrieved_results(multiheirtt_task_result, method_name = 'hippo_rag_baseline')


# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load()
vector_store = get_vector_store()
FinQA_hippo_rag = HippoRAG(retriever = vector_store)
FinQA_hippo_rag.offline_indexing(corpus = finqa_task.corpus, batch_size = 8)
finqa_task_result = FinQA_hippo_rag.retrieve(queries = finqa_task.queries, corpus = finqa_task.corpus, top_k = 10)
# finqa_task_reranker = CrossEncoderReranker(queries = finqa_task.queries, corpus = finqa_task.corpus, reranker = model)
# finqa_task_final_result = finqa_task_reranker.rerank(retrieved_result = finqa_task_result, top_k = 10)
finqa_task.save_retrieved_results(finqa_task_result, method_name = 'hippo_rag_baseline')


# FinanceBench Task
print(f"FinanceBench task")
financebench_task = FinanceBenchTask()
financebench_task.load()
vector_store = get_vector_store()
FinanceBench_hippo_rag = HippoRAG(retriever = vector_store)
FinanceBench_hippo_rag.offline_indexing(corpus = financebench_task.corpus, batch_size = 8)
financebench_task_result = FinanceBench_hippo_rag.retrieve(queries = financebench_task.queries, corpus = financebench_task.corpus, top_k = 10)
# financebench_task_reranker = CrossEncoderReranker(queries = financebench_task.queries, corpus = financebench_task.corpus, reranker = model)
# financebench_task_final_result = financebench_task_reranker.rerank(retrieved_result = financebench_task_result, top_k = 10)
financebench_task.save_retrieved_results(financebench_task_result, method_name = 'hippo_rag_baseline')


# ConvFinQA Task
print(f"ConvFinQA task")
convfinqa_task = ConvFinQATask()
convfinqa_task.load()
vector_store = get_vector_store()
ConvFinQA_hippo_rag = HippoRAG(retriever = vector_store)
ConvFinQA_hippo_rag.offline_indexing(corpus = convfinqa_task.corpus, batch_size = 8)
convfinqa_task_result = ConvFinQA_hippo_rag.retrieve(queries = convfinqa_task.queries, corpus = convfinqa_task.corpus, top_k = 10)
# convfinqa_task_reranker = CrossEncoderReranker(queries = convfinqa_task.queries, corpus = convfinqa_task.corpus, reranker = model)
# convfinqa_task_final_result = convfinqa_task_reranker.rerank(retrieved_result = convfinqa_task_result, top_k = 10)
convfinqa_task.save_retrieved_results(convfinqa_task_result, method_name = 'hippo_rag_baseline')


# FinQABench Task
print(f"FinQABench task")
finqabench_task = FinQABenchTask()
finqabench_task.load()
vector_store = get_vector_store()
FinQABench_hippo_rag = HippoRAG(retriever = vector_store)
FinQABench_hippo_rag.offline_indexing(corpus = finqabench_task.corpus, batch_size = 8)
finqabench_task_result = FinQABench_hippo_rag.retrieve(queries = finqabench_task.queries, corpus = finqabench_task.corpus, top_k = 10)
# finqabench_task_reranker = CrossEncoderReranker(queries = finqabench_task.queries, corpus = finqabench_task.corpus, reranker = model)
# finqabench_task_final_result = finqabench_task_reranker.rerank(retrieved_result = finqabench_task_result, top_k = 10)
finqabench_task.save_retrieved_results(finqabench_task_result, method_name = 'hippo_rag_baseline')


# TATQA Task
print(f"TATQA task")
tatqa_task = TATQATask()
tatqa_task.load()
vector_store = get_vector_store()
TATQA_hippo_rag = HippoRAG(retriever = vector_store)
TATQA_hippo_rag.offline_indexing(corpus = tatqa_task.corpus, batch_size = 8)
tatqa_task_result = TATQA_hippo_rag.retrieve(queries = tatqa_task.queries, corpus = tatqa_task.corpus, top_k = 10)
# tatqa_task_reranker = CrossEncoderReranker(queries = tatqa_task.queries, corpus = tatqa_task.corpus, reranker = model)
# tatqa_task_final_result = tatqa_task_reranker.rerank(retrieved_result = tatqa_task_result, top_k = 10)
tatqa_task.save_retrieved_results(tatqa_task_result, method_name = 'hippo_rag_baseline')


# FinDER Task
print(f"FinDER task")
finder_task = FinDERTask()
finder_task.load()
vector_store = get_vector_store()
FinDER_hippo_rag = HippoRAG(retriever = vector_store)
FinDER_hippo_rag.offline_indexing(corpus = finder_task.corpus, batch_size = 8)
finder_task_result = FinDER_hippo_rag.retrieve(queries = finder_task.queries, corpus = finder_task.corpus, top_k = 10)
# finder_task_reranker = CrossEncoderReranker(queries = finder_task.queries, corpus = finder_task.corpus, reranker = model)
# finder_task_final_result = finder_task_reranker.rerank(retrieved_result = finder_task_result, top_k = 10)
finder_task.save_retrieved_results(finder_task_result, method_name = 'hippo_rag_baseline')

MultiHeirtt task


Loading:   0%|          | 1/1310 [02:27<53:30:27, 147.16s/it]


KeyboardInterrupt: 

In [ ]:
xinchaof tat ca  

5

In [ ]:
financebench_task = FinanceBenchTask()
financebench_task.load()

In [ ]:
query_ids = []
with open(f"process.log", 'r') as f:
    lines = f.readlines()
    for line in lines:
        query_id = re.findall(pattern='qd.*\d$', string = line)
        if query_id:
            query_ids.extend(query_id)

query_ids = set(query_ids)
query_ids

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
for query_id in query_ids:
    query = financebench_task.queries[query_id]
    doc = nlp(query)
    print(f"Query_id {query_id}: Entities = {list(doc.ents).map(str)}")

In [ ]:
financebench_task.queries['qd2abc1e6']

In [ ]:
docs = nlp(financebench_task.queries['qd2abc1e6'])

In [ ]:
docs.ents

In [ ]:
# Next step:
# 1. Replac

In [ ]:
hippo_rag.ner_extractor(financebench_task.queries['qd2abbd7c'])

In [ ]:
financebench_task.queries['qd2abbd7c']

In [ ]:
g.get_eid('nommi', 'khai', error = False)

In [ ]:
g.vs['name']

In [ ]:
from langchain_core.output_parsers import ListOutputParser
import re
class CustomizedListParser(ListOutputParser):
    def parse(self, text) -> List[List[List[str]]]:
        pattern = r"\[\s*(?:\[\s*(?:\[[^\]]*?\]\s*,?\s*\n*)*\s*\]\s*,?\s*)*\s*\]"
        result = re.findall(pattern, text)
        if result:
            extracted_text = re.findall(pattern, text)[0]
            return eval(extracted_text)
        return []

In [ ]:
test_str = """

[
 [["Radio City", "located in", "India"],
 ["Radio City", "is", "private FM radio station"],
 ["Radio City", "started on", "3 July 2001"],
 ["Radio City", "plays songs in", "Hindi"],
 ["Radio City", "plays songs in", "English"],
 ["Radio City", "forayed into", "New Media"],
 ["Radio City", "launched", "PlanetRadiocity.com"],
 ["PlanetRadiocity.com", "launched in", "May 2008"],
 ["PlanetRadiocity.com", "is", "music portal"],
 ["PlanetRadiocity.com", "offers", "news"],
 ["PlanetRadiocity.com", "offers", "videos"],
 ["PlanetRadiocity.com", "offers", "songs"]]
 ]
 """

CustomizedListParser().parse(test_str * 100000)


In [ ]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    {task_variable}_final_result = extract_result({task_variable}_result, k = 10)
    {task_variable}.save_retrieved_results({task_variable}_final_result, method_name = 'dense_retrieval_split_only')
    """
    print(script_string)

In [ ]:
def extract_result(task_result : Dict[str, Dict[str, float]], k = 10):
    final_result = {}
    for query_id, doc_dict in task_result.items():
        final_result[query_id] = {}
        for i, (corpus_id, score) in enumerate(doc_dict.items()):
            if (i == k): break
            final_result[query_id][corpus_id] = score
    return final_result

In [ ]:

# # FinQA Task
# finqa_task_final_result = extract_result(finqa_task_result, k = 10)
# finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_split_only')


# # FinanceBench Task
# financebench_task_final_result = extract_result(financebench_task_result, k = 10)
# financebench_task.save_retrieved_results(financebench_task_final_result, method_name = 'dense_retrieval_split_only')


# # ConvFinQA Task
# convfinqa_task_final_result = extract_result(convfinqa_task_result, k = 10)
# convfinqa_task.save_retrieved_results(convfinqa_task_final_result, method_name = 'dense_retrieval_split_only')


# # FinQABench Task
# finqabench_task_final_result = extract_result(finqabench_task_result, k = 10)
# finqabench_task.save_retrieved_results(finqabench_task_final_result, method_name = 'dense_retrieval_split_only')


# # TATQA Task
# tatqa_task_final_result = extract_result(tatqa_task_result, k = 10)
# tatqa_task.save_retrieved_results(tatqa_task_final_result, method_name = 'dense_retrieval_split_only')


# FinDER Task
finder_task_final_result = extract_result(finder_task_result, k = 10)
finder_task.save_retrieved_results(finder_task_final_result, method_name = 'dense_retrieval_split_only')

In [ ]:





# TATQA Task
tatqa_task_final_result = extract_result(tatqa_task_result, k = 10)
tatqa_task.save_retrieved_results(tatqa_task_final_result, method_name = 'dense_retrieval_only')


# FinDER Task
finder_task_final_result = extract_result(finder_task_result, k = 10)
finder_task.save_retrieved_results(finder_task_final_result, method_name = 'dense_retrieval_only')

In [ ]:
method_name = 'dense_retrieval_split_document_and_reranking'
final_result = get_final_result(dataset_names, method_name = method_name)

In [ ]:
final_result

In [ ]:
final_result

In [ ]:
final_result.to_csv(f'submission_{method_name}.csv', index = False)

In [ ]:
import torch.nn as nn

In [ ]:

# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load()
vector_store = get_vector_store()
finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
finqa_task_retriever.load_corpus_without_splitting(corpus = finqa_task.corpus, saved_index = False)
finqa_task_result = finqa_task.retrieve(retriever = finqa_task_retriever, top_k = 50)
finqa_task_reranker = CrossEncoderReranker(queries = finqa_task.queries, corpus = finqa_task.corpus, reranker = model)
finqa_task_final_result = finqa_task_reranker.rerank(finqa_task_result, top_k = 10)
finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_and_reranking')





In [ ]:
# # MultiHeirtt Task
# multiheirtt_task_final_result = extract_result(multiheirtt_task_result, k = 10)
# multiheirtt_task.save_retrieved_results(multiheirtt_task_final_result, method_name = 'dense_retrieval_only')


# FinQA Task
finqa_task_final_result = extract_result(finqa_task_result, k = 10)
finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_only')





In [ ]:
a = {'s': {'s' : 1}}
isinstance(a, Dict)